In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType
from pyspark.sql import functions as F
catalog_name ="ecommerce"

df_gold=spark.table(f"{catalog_name}.silver.slv_order_items")
display(df_gold)

df_gold=df_gold.withColumn("gross_amount",F.col("quantity")*F.col("unit_price"))

df_gold=df_gold.withColumn("discount_amount",F.ceil(F.col("gross_amount")*(F.col("discount_pct")/100.0))
)

df_gold=df_gold.withColumn("sale_amount",F.col("gross_amount")-F.col("discount_amount")+F.col("tax_amount"))

df_gold=df_gold.withColumn("date_id",F.date_format(F.col("dt"),"yyyyMMdd").cast(IntegerType()))

In [0]:
fx_rates = {
    "INR": 1.00,
    "AED": 24.18,
    "AUD": 57.55,
    "CAD": 62.93,
    "GBP": 117.98,
    "SGD": 68.18,
    "USD": 88.29,
}

rates = [[k, float(v)] for k, v in fx_rates.items()]
rates_df = spark.createDataFrame(rates, ["currency", "inr_rate"])

df_gold = (
    df_gold
    .join(
        rates_df,
        rates_df.currency == F.upper(F.trim(F.col("unit_price_currency"))),
        "left"
    )
    .withColumn("sale_amount_inr", F.col("sale_amount") * rates_df["inr_rate"])
    .drop(rates_df.currency, rates_df.inr_rate)
    .withColumn("sale_amount_inr", F.ceil(F.col("sale_amount_inr")))
)

df_gold.printSchema()

In [0]:
orders_gold_df = df_gold.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sale_amount").alias("net_amount"),
    F.col("sale_amount_inr").alias("net_amount_inr")
)

orders_gold_df.printSchema()


In [0]:
# Write raw data to the gold layer (catalog: ecommerce, schema: gold, table: gld_fact_order_items)
orders_gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")